# Basic Data Analysis of the NASA Milling Dataset

In [ ]:
# Imports
import scipy.io
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, interactive, fixed, interact_manual, widgets

In [ ]:
# Load from parquet file
nasa_df = pd.read_parquet('/Users/flore/Documents/01_Forschung/09_Coding.nosync/07_Tool_wear_DA/tool-wear-domain-adaption/data/nasa_milling_normalized.parquet')
nasa_df.head()

In [ ]:
#Visualize the wear ('VB') over time ('run') for each case ('case')
import seaborn as sns
plt.figure(figsize=(12, 6))
sns.lineplot(data=nasa_df, x='run', y='wear_norm', hue='CaseID', marker='o')
plt.title('Tool Wear Over Time for Each Case')
plt.xlabel('Cut Number')
plt.ylabel('Normalized Tool Wear (1 = No Wear, 0 = Max Wear)')
plt.legend(title='Case ID', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid()
plt.tight_layout()
plt.show()

## Data Visualization

#### Do some simple Plotting of the data

In [ ]:
# Create a histogram for the 'VB' column from the settings_DF DataFrame.
# The hist() function automatically ignores NaN values.
plt.hist(nasa_df['wear_norm'].dropna(), bins=20, edgecolor='black')
plt.title('Distribution of Flank Wear (VB)')
plt.xlabel('VB (mm)')
plt.ylabel('Frequency')
plt.grid(axis='y', alpha=0.75)
plt.show()

#### Look at a single cut

There are 167 cases in the data set. Switching those numbers manually in the code above will show the data for each case. But this is not very elegant.
So lets built a quick interactive solution for this. 

In [ ]:
# Define the function to plot signals for a given case
def plot_signals_for_case(case_id):
    case_data = nasa_df[nasa_df['CaseID'] == case_id]
    
    if case_data.empty:
        print(f"No data found for Case ID: {case_id}")
        return

    # Get signal columns (those with array data)
    signal_cols = [col for col in case_data.columns if isinstance(case_data[col].iloc[0], np.ndarray)]
    
    if not signal_cols:
        print("No signal data with arrays found for this case.")
        return
        
    for signal in signal_cols:
        plt.figure(figsize=(15, 4))
        ax = plt.gca()
        
        # Plot each cut's signal array for the current signal type
        for idx, row in case_data.iterrows():
            # Check if data is a numpy array and not empty
            if isinstance(row[signal], np.ndarray) and row[signal].size > 0:
                ax.plot(row[signal], label=f"Cut {row['run']}")
        
        ax.set_title(f'{signal} for Case {case_id}')
        ax.set_ylabel(signal)
        ax.set_xlabel('Time Step')
        ax.legend(title='Cut', bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()

# Get unique case IDs
case_ids = sorted(nasa_df['CaseID'].unique())

# Create an interactive dropdown widget
interact(plot_signals_for_case, case_id=widgets.Dropdown(options=case_ids, description='Case ID:'));

#### Interactive Plot with VB Filter
Here is a more advanced interactive plot. You can select a range for `VB` (wear) and decide whether to include cuts where `VB` is not available (`NaN`). The available `cut_no` values will be updated based on your selection.

### Remove outliers 
If we go through the cuts, we spot two outliers, that should be erased for cut_no: 17 and 94. These would otherwise moderate our statistics.

In [ ]:
nasa_df.drop(nasa_df.loc[nasa_df['run'].isin([17,94])].index, inplace=True)

### More Statistics

Let's first start with some boxplots. of the signals. We keep the "grouping" widgets from the settings df from above.